In [33]:
# This file is a modification of animal_class_classifier.ipynb.
# The main difference is that I perform a train-test split on the larger audio files before breaking them into 1-second segments.

import librosa
import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [34]:
train_soundscapes_labels_df = pd.read_csv("train_soundscapes_labels.csv")

train_soundscapes_reduced_df = pd.DataFrame({
    "filename": train_soundscapes_labels_df.iloc[:, :2].astype(str).agg(''.join, axis=1),
    "primary_label": train_soundscapes_labels_df.iloc[:, 3]
})

In [35]:
train_soundscapes_labels_df.head()

,filename,start,end,primary_label
0,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:00,00:00:05,22961;23158;24321;517063;65380
1,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:05,00:00:10,22961;23158;24321;517063;65380
2,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:10,00:00:15,22961;23158;24321;517063;65380
3,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:15,00:00:20,22961;23158;24321;517063;65380
4,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:20,00:00:25,22961;23158;24321;517063;65380


In [36]:
train_soundscapes_reduced_df.head()

,filename,primary_label
0,BC2026_Train_0039_S22_20211231_201500.ogg00:00:00,22961;23158;24321;517063;65380
1,BC2026_Train_0039_S22_20211231_201500.ogg00:00:05,22961;23158;24321;517063;65380
2,BC2026_Train_0039_S22_20211231_201500.ogg00:00:10,22961;23158;24321;517063;65380
3,BC2026_Train_0039_S22_20211231_201500.ogg00:00:15,22961;23158;24321;517063;65380
4,BC2026_Train_0039_S22_20211231_201500.ogg00:00:20,22961;23158;24321;517063;65380


In [37]:
labels_multi = []
for filename in train_soundscapes_labels_df["filename"]:
    labels_multi += (filename not in labels_multi)*[filename]

In [38]:
len(labels_multi)

66

In [39]:
# train_audio has 206 folders, each folder corresponds to a primary label.

folder_path = "train_audio"
labels_single = []

for item in os.listdir(folder_path):
    labels_single += [item]

In [40]:
len(labels_single)

207

In [41]:
# Performs train-test splits on both sets of labels.
# We shuffle the indices to ensure that the training and testing sets are randomly distributed.
# We will then replace the labels with the corresponding spectrograms.

indices_multi = np.arange(len(labels_multi))
np.random.shuffle(indices_multi)
train_indices_multi, test_indices_multi = train_test_split(indices_multi, test_size=0.2, random_state=42)

indices_single = np.arange(len(labels_single))
np.random.shuffle(indices_single)
train_indices_single, test_indices_single = train_test_split(indices_single, test_size=0.2, random_state=42)

In [42]:
train_indices_multi[:5], test_indices_multi[:5], train_indices_single[:5], test_indices_single[:5]

(array([48, 23, 29, 41, 21]),
 array([44, 14,  1, 53, 47]),
 array([102, 188,  95,  13,  32]),
 array([ 54,   6, 195,  47, 143]))

In [43]:
X_labels_multi_train = [labels_multi[i] for i in train_indices_multi]
X_labels_multi_test = [labels_multi[i] for i in test_indices_multi]
X_labels_single_train = [labels_single[i] for i in train_indices_single]
X_labels_single_test = [labels_single[i] for i in test_indices_single]

In [44]:
X_labels_multi_train[:5], X_labels_multi_test[:5], X_labels_single_train[:5], X_labels_single_test[:5]

(['BC2026_Train_0041_S22_20220105_223000.ogg',
  'BC2026_Train_0034_S22_20211222_013000.ogg',
  'BC2026_Train_0029_S22_20211211_023000.ogg',
  'BC2026_Train_0012_S03_20220211_233000.ogg',
  'BC2026_Train_0025_S22_20211127_024500.ogg'],
 ['BC2026_Train_0002_S08_20250607_030007.ogg',
  'BC2026_Train_0044_S22_20220124_023000.ogg',
  'BC2026_Train_0019_S22_20211104_200000.ogg',
  'BC2026_Train_0010_S09_20250828_000000.ogg',
  'BC2026_Train_0063_S19_20241214_190000.ogg'],
 ['chvcon1', '24279', 'ruftof1', '22967', 'tattin1'],
 ['blheag1', 'magtan2', '47144', 'rufgna3', 'whwpic1'])

In [45]:
len(X_labels_multi_train), len(X_labels_multi_test), len(X_labels_single_train), len(X_labels_single_test)

(52, 14, 165, 42)

In [46]:
# Loading the folder

train_audio_df = pd.read_csv("train_audio_df.csv")

In [47]:
count = 0
for row in train_audio_df.itertuples():
    print(row.filename.split("/")[0])
    count += 1
    if count == 10:
        break

crebec1
crebec1
crebec1
crebec1
crebec1
crebec1
crebec1
crebec1
crebec1
crebec1


In [48]:
# This block classifies the .ogg files in train_audio_df.csv as either training or testing data,
# based on the train-test split we just performed on the labels.
# The .ogg files are the 5-second segments that we will create later from the longer audio files.
# This code takes 16 minutes to run.

X_single_train = []
y_single_train = []
X_single_test = []
y_single_test = []

NUM_CLASSES = 5
SR = 32000
N_MELS = 128
EXPECTED_FRAMES = int(np.ceil((SR * 5) / 512))
classes = ['Insecta', 'Reptilia', 'Amphibia', 'Mammalia', 'Aves']

for row in train_audio_df.itertuples():
    filename = row.filename
    
    audio, _ = librosa.load(
        f"train_audio/{filename}",
        sr=SR
    )

# After breaking the audio files into 5 second chunks, there were far too many files.
# Rather than processing every file, we take a few chunks.
# If the file is less than 15 seconds, we take 1 chunk. Otherwise, we take 2.

    chunk_length = 5 * SR

    if len(audio) < 2 * chunk_length:
        # 5–10 sec → just take first chunk
        audio_clips = [audio[:chunk_length]]

    elif len(audio) < 3 * chunk_length:
        # 10–15 sec → allow overlap (random is fine)
        audio_clips = []
        for _ in range(2):
            start = np.random.randint(0, len(audio) - chunk_length)
            audio_clips.append(audio[start:start + chunk_length])

    else:
        # ≥15 sec → enforce separation safely
        start1 = np.random.randint(0, len(audio) - chunk_length)
        start2 = np.random.randint(0, len(audio) - chunk_length)

        # Try a few times only (avoid slow loops)
        for _ in range(5):
            if abs(start1 - start2) >= chunk_length:
                break
            start2 = np.random.randint(0, len(audio) - chunk_length)

        audio_clips = [audio[start1:start1 + chunk_length], audio[start2:start2 + chunk_length]]

# Converts the audio files in audio_clips to a mel spectrogram, and normalizes it to the range [0, 1].

    for audio_clip in audio_clips:

        mel = librosa.feature.melspectrogram(
            y=audio_clip,
            sr=SR,
            n_mels=N_MELS
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())

        # Pad / trim
        if mel_db.shape[1] < EXPECTED_FRAMES:
            mel_db = np.pad(mel_db, ((0, 0), (0, EXPECTED_FRAMES - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :EXPECTED_FRAMES]

        label_vector = [0] * NUM_CLASSES
        for j, cls in enumerate(classes):
            if row.primary_label == cls:
                label_vector[j] = 1


        if row.filename.split('/')[0] in X_labels_single_train:
            X_single_train.append(mel_db[..., np.newaxis])
            y_single_train.append(label_vector)

        else:
            X_single_test.append(mel_db[..., np.newaxis])
            y_single_test.append(label_vector)

X_single_train = np.array(X_single_train, dtype=np.float32)
y_single_train = np.array(y_single_train, dtype=np.float32)
X_single_test = np.array(X_single_test, dtype=np.float32)
y_single_test = np.array(y_single_test, dtype=np.float32)

In [49]:
len(X_single_train), len(y_single_train), len(X_single_test), len(y_single_test)

(48446, 48446, 12070, 12070)

In [50]:
# Organizes the files in train_audio_df.csv into the 5 classes. These files are usually multi-class.

taxonomy_df = pd.read_csv("taxonomy.csv")
classes = ['Insecta', 'Reptilia', 'Amphibia', 'Mammalia', 'Aves']
filenames = {value: [] for value in classes}

for entry in train_soundscapes_reduced_df.itertuples():
    filename = entry.filename
    primary_label = entry.primary_label.split(';')

    found = {value: False for value in classes}
    
    for label in primary_label:
        for value in classes:
            if taxonomy_df[taxonomy_df['primary_label'] == label]['class_name'].values[0] == value:
                found[value] = True
    
    for value in classes:
         if found[value]:
            filenames[value].append(filename)

In [51]:
X_multi_train = []
y_multi_train = []
X_multi_test = []
y_multi_test = []

# This loop converts each audio file in train_soundscapes into a mel spectrogram.

for filename in train_soundscapes_reduced_df['filename']:
    file = filename[:-8]
    start = int(filename[-2:])
    
    audio, _ = librosa.load(
        f"train_soundscapes/{file}",
        sr=SR,
        offset=start,
        duration=5
    )

# Converts the audio to a mel spectrogram, and normalizes it to the range [0, 1].

    mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())

    # 🔥 MULTI-LABEL TARGET
    label_vector = [0] * NUM_CLASSES
    for i, cls in enumerate(classes):
        if filename in filenames[cls]:
            label_vector[i] = 1

    if file in X_labels_multi_train:
        X_multi_train.append(mel_db[..., np.newaxis])
        y_multi_train.append(label_vector)
    else:
        X_multi_test.append(mel_db[..., np.newaxis])
        y_multi_test.append(label_vector)
    
X_multi_train = np.array(X_multi_train, dtype=np.float32)
y_multi_train = np.array(y_multi_train, dtype=np.float32)
X_multi_test = np.array(X_multi_test, dtype=np.float32)
y_multi_test = np.array(y_multi_test, dtype=np.float32)

# X is the list of spectrograms, and y is the list of multi-label vectors.

In [52]:
len(X_multi_train), len(y_multi_train), len(X_multi_test), len(y_multi_test)

(1162, 1162, 316, 316)

In [53]:
X_train = np.concatenate([X_multi_train, X_single_train], axis=0)
X_test = np.concatenate([X_multi_test, X_single_test], axis=0)
y_train = np.concatenate([y_multi_train, y_single_train], axis=0)
y_test = np.concatenate([y_multi_test, y_single_test], axis=0)

In [54]:
# Now that we have all the files, we can make a model.
# In a previous version of the code, we showed that the class weights of each class in our model are as follows.

class_weights = {'Insecta' : 0.8534296, 'Reptilia' : 11.257143, 'Amphibia' : 0.26092714, 'Mammalia' : 4.075862, 'Aves' : 0.7141994}
class_weights_array = np.array([class_weights[cls] for cls in class_weights])

In [55]:
def weighted_bce(y_true, y_pred):
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)

    weights = y_true * class_weights_array + (1 - y_true)

    return tf.reduce_mean(bce * weights, axis=-1)

In [56]:
# We have far more single class data than multi-class data. An unweighted algorithm will say that a spectrogram is single class far too often.
# This block weights the train and test sets so that the model gets enough multi-class data.

multi_idx = np.where(y_train.sum(axis=1) > 1)[0]
single_idx = np.where(y_train.sum(axis=1) == 1)[0]

# repeat multi-label samples
# mild oversampling
oversampled_idx = np.concatenate([
    single_idx,
    np.repeat(multi_idx, 3)
])

# plus weighting
sample_weights = np.where(y_train.sum(axis=1) > 1, 2.0, 1.0)

np.random.shuffle(oversampled_idx)

X_train = X_train[oversampled_idx]
y_train = y_train[oversampled_idx]

In [57]:
X = np.concatenate([X_train, X_test], axis=0)
y = np.concatenate([y_train, y_test], axis=0)

In [58]:
input_shape = X.shape[1:]  # (128, frames, 1)

NUM_CLASSES = 5

model = models.Sequential([
    layers.Input(shape=input_shape),
    
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(NUM_CLASSES, activation='sigmoid')  # 🔥 multi-label
])

In [59]:
def weighted_bce(y_true, y_pred):
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
    weights = y_true * class_weights_array + (1 - y_true)
    return tf.reduce_mean(bce * weights, axis=-1)

In [60]:
# Compiles the model.

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss=weighted_bce,
    metrics=[
        tf.keras.metrics.AUC(curve="PR", multi_label=True, name="pr_auc"),
        tf.keras.metrics.AUC(curve="ROC", multi_label=True, name="roc_auc"),
        tf.keras.metrics.Precision(thresholds=0.3),
        tf.keras.metrics.Recall(thresholds=0.3)
    ]
)

In [61]:
# Sanity check

class_weights = {'Insecta' : 0.8534296, 'Reptilia' : 11.257143, 'Amphibia' : 0.26092714, 'Mammalia' : 4.075862, 'Aves' : 0.7141994}
class_weights_array = np.array([class_weights[cls] for cls in class_weights])
class_weights_array.shape == (NUM_CLASSES,)

True

In [62]:
sample_weights = np.where(y_train.sum(axis=1) > 1, 2.0, 1.0)
len(sample_weights) == len(X_train)

True

In [63]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_animal_model.keras",
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        mode="min",
        verbose=1
    )
]

In [64]:
# Trains the model
# This block took 3 hours on a CPU. The model had 97.5% precision and 98.6% recall.

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=64,
    shuffle=True,
    callbacks = callbacks
)

Epoch 1/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 0s 851ms/step - loss: 0.5621 - pr_auc: 0.2195 - precision_1: 0.2821 - recall_1: 0.9136 - roc_auc: 0.6264
Epoch 1: val_loss improved from None to 0.18341, saving model to best_animal_model.keras

Epoch 1: finished saving model to best_animal_model.keras
787/787 ━━━━━━━━━━━━━━━━━━━━ 712s 900ms/step - loss: 0.3720 - pr_auc: 0.2258 - precision_1: 0.3915 - recall_1: 0.9450 - roc_auc: 0.6929 - val_loss: 0.1834 - val_pr_auc: 0.3323 - val_precision_1: 0.7673 - val_recall_1: 0.9715 - val_roc_auc: 0.7372 - learning_rate: 3.0000e-04
Epoch 2/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 0s 873ms/step - loss: 0.0883 - pr_auc: 0.3812 - precision_1: 0.9168 - recall_1: 0.9618 - roc_auc: 0.8471
Epoch 2: val_loss improved from 0.18341 to 0.07814, saving model to best_animal_model.keras

Epoch 2: finished saving model to best_animal_model.keras
787/787 ━━━━━━━━━━━━━━━━━━━━ 718s 911ms/step - loss: 0.0788 - pr_auc: 0.4054 - precision_1: 0.9274 - recall_1: 0.9621 - roc_auc: 0.8625 

In [44]:
model.save("best_multilabel_model.keras")